# README

- **Author**: 李芸榛
- **Created At**: 2026-11-10
- **Last Modified At**: 2026-11-10

---

## What does this file do?

- 由 v7 交易資料計算臺大周邊 YouBike 站點間的**條件轉移機率** $P(\text{還車站}=j \mid \text{借車站}=i,\ \text{時期}=p,\ \text{星期}=w,\ \text{借車小時}=h)$，作為馬可夫鏈模擬車輛流動的轉移矩陣；並計算每站每小時的平均借出量，作為模擬時的借車到達率。
- **觀察單位**：輸入為一筆旅次；輸出 1 為 (時期, 星期, 小時, 借車站, 還車站) 一列，輸出 2 為 (時期, 星期, 小時, 借車站) 一列。
- **時期 p**：以 `infodate`（與借車日期全部相同，已核對）對應日曆的 `semester`：1 =「學期」、0 =「假期」。日曆的學期旗標已於 2026-11-10 延伸到開學前 2 天與期末後 7 天（見 [calendar_semester_buffer.R](calendar_semester_buffer.R)）。
- **星期 w**：日曆 `day_of_week`（1 = 星期一 … 7 = 星期日）；**小時 h**：`rent_time` 的小時（資料只精確到小時），跨日旅次以借車時間為準。
- **站點（組員 2026-11-10 確認）**：站點 ID 一律以文字讀入。範圍內 109 站以 `station_id` 辨識（已確認沒有一個 ID 對應多個站名）。`station_id = -1`（範圍外）合併為「區外」，**只作為終點，不作為起點**，因為資料只收錄至少一端在範圍內的旅次，「區外 → 區外」不可能被觀察到。所以轉移矩陣是 109 × 110（列 = 109 個範圍內借車站，欄 = 109 站 + 區外），每列加總為 1；從範圍外借車的旅次（`trip_group = a`）不進入計算。同站借還（i = j）保留。
- **一般車與電輔車合併計算**（組員確認）。
- **排除規則**，依序為：
  1. 同站借還且騎乘時間（`rent`）≤ 1 分鐘：視為借車失敗。`rent` 只用於這項篩選。
  2. 借車站在範圍外（`rent_station_id = -1`）。
  3. 特殊日期：星期一到五遇到放假（`weekday = 0`）以及週六、日補班（`weekday = 1`），通勤模式與同一個星期幾不同，整天排除。
- **樣本數門檻**：某個 (p, w, h, i) 的借出總筆數 < 30 時，該列 `prob` 為 NA、`below_threshold = True`。備援機率 `prob_fallback` 以 (p, 平日／週末, h, i) 計算（平日 = 一到五、週末 = 六、日），同樣使用 30 筆門檻。
- **借車到達率**：`mean_rentals_per_day` = 總借出筆數 ÷ `n_days`。`n_days` 為該 (時期, 星期) 扣除特殊日期後、**該站已營運**的天數（組員確認）。營運起日 = 每小時流量檔 `in_service` 第一次為 1 的日期與該站第一筆交易日期取較晚者，用來處理 2024–2025 年間才啟用的站，以及標為啟用但到 2024-06 才有交易的臺大共同教室北側。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet`
     - Input key: `traffic_2024`
     - Description: 2024 年借或還任一端在研究範圍內的 YouBike 2.0 交易，一列一筆；本檔使用 `rent_time`、`rent_station`、`rent_station_id`、`return_station`、`return_station_id`、`rent`（騎乘時間 HH:MM:SS）、`infodate`。
     - Documentation: `data/processed/youbike_traffic/YouBike_traffic_doc.ipynb`
     - SHA-256: `5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119`
  2. `data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet`
     - Input key: `traffic_2025`
     - Description: 同上，2025 年。
     - Documentation: `data/processed/youbike_traffic/YouBike_traffic_doc.ipynb`
     - SHA-256: `9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9`
  3. `data/processed/calendar_2024.csv`
     - Input key: `calendar_2024`
     - Description: 2024 年每日一列：`date`、`day_of_week`、`weekday`（上班日 = 1）、`semester`（學期 = 1，含開學前 2 天與期末後 7 天）、`day_off_reason`。
     - Documentation: processed-data 文件待建立；產生方式見 `data/code/youbike_flow_hourly.Rmd`、組員手動修正與 `data/code/calendar_semester_buffer.R`
     - SHA-256: `0625623f24d1e21b3f9a9877e486eec6a1703da8891337485e5e52b88335aaee`
  4. `data/processed/calendar_2025.csv`
     - Input key: `calendar_2025`
     - Description: 同上，2025 年。
     - Documentation: 同上
     - SHA-256: `91a87c16760043bd57b3e15fe85945ce2ebabf8f3b2b9ddca5781a72783553c4`
  5. `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet`
     - Input key: `hourly_flow`
     - Description: 109 站 × 每小時的借還量與 `in_service`；本檔只用 `station_id`、`datetime`、`in_service` 判定各站營運起日。
     - Documentation: `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_doc.ipynb`
     - SHA-256: `3db6469dc42efb8dc7fa24661096de56b7b6fd71465d3fd9f3214767daff41ca`

---

## What does this file output?

- `data/processed/youbike_transition/transition_probs.parquet`
  - Output key: `transition_probs`
  - Description: 條件轉移機率長表。欄位 `period`, `day_of_week`, `hour`, `origin_id`, `origin_name`, `dest_id`, `dest_name`, `n_trips`, `n_origin_total`, `prob`, `below_threshold`, `prob_fallback`。只存「在對應的平日／週末備援格中至少出現一次」的 (i, j) 組合；同一列中沒出現的終點機率為 0（`get_matrix` 會補 0）。
  - Role: `analysis`（馬可夫鏈轉移矩陣）
  - Documentation: 本 notebook（processed-data 文件待建立）；讀取函式見 [transition_matrix.py](transition_matrix.py)
  - Expected schema and row-count rules: ID、站名為 string；`period` 為 category（學期、假期）；`day_of_week`、`hour` 為 int8；`n_trips`、`n_origin_total` 為 int32；`prob`、`prob_fallback` 為 float64；`below_threshold` 為 bool。(period, day_of_week, hour, origin_id, dest_id) 唯一；未標 NA 的 (p, w, h, i) 機率加總 = 1。
  - Verification status / SHA-256: 見下方讀回結果
  - Downstream finding input: 同一檔案
- `data/processed/youbike_transition/rent_rates.parquet`
  - Output key: `rent_rates`
  - Description: 每站每小時平均借出量。欄位 `period`, `day_of_week`, `hour`, `origin_id`, `n_trips_total`, `n_days`, `mean_rentals_per_day`；涵蓋全部 2 × 7 × 24 × 109 = 36,624 個組合（沒有借車的組合為 0）。
  - Role: `analysis`（模擬的借車到達率）
  - Documentation: 本 notebook（processed-data 文件待建立）
  - Expected schema and row-count rules: 36,624 列、(period, day_of_week, hour, origin_id) 唯一；型態同上，`n_days` 為 int32、`mean_rentals_per_day` 為 float64。
  - Verification status / SHA-256: 見下方讀回結果
  - Downstream finding input: 同一檔案

All paths are relative to the project root, not the operating-system root.

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual

In [2]:
INPUTS = {
    "traffic_2024": "data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet",
    "traffic_2025": "data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet",
    "calendar_2024": "data/processed/calendar_2024.csv",
    "calendar_2025": "data/processed/calendar_2025.csv",
    "hourly_flow": "data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet",
}
EXPECTED_INPUT_SHA256 = {
    "traffic_2024": "5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119",
    "traffic_2025": "9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9",
    "calendar_2024": "0625623f24d1e21b3f9a9877e486eec6a1703da8891337485e5e52b88335aaee",
    "calendar_2025": "91a87c16760043bd57b3e15fe85945ce2ebabf8f3b2b9ddca5781a72783553c4",
    "hourly_flow": "3db6469dc42efb8dc7fa24661096de56b7b6fd71465d3fd9f3214767daff41ca",
}
OUTPUTS = {
    "transition_probs": "data/processed/youbike_transition/transition_probs.parquet",
    "rent_rates": "data/processed/youbike_transition/rent_rates.parquet",
}
TRAFFIC_COLUMNS = ["rent_time", "rent_station", "rent_station_id", "return_station",
                   "return_station_id", "rent", "infodate"]
# Station IDs and all text fields are read as strings, never as numbers.
INPUT_READERS = {
    "traffic_2024": lambda p: pd.read_parquet(p, engine="pyarrow", columns=TRAFFIC_COLUMNS),
    "traffic_2025": lambda p: pd.read_parquet(p, engine="pyarrow", columns=TRAFFIC_COLUMNS),
    "calendar_2024": lambda p: pd.read_csv(p, dtype=str, keep_default_na=False),
    "calendar_2025": lambda p: pd.read_csv(p, dtype=str, keep_default_na=False),
    "hourly_flow": lambda p: pd.read_parquet(p, engine="pyarrow",
                                             columns=["station_id", "datetime", "in_service"]),
}

input_paths = {}
for key, relative_path in INPUTS.items():
    area = Path(relative_path).parts[1] if len(Path(relative_path).parts) > 1 else ""
    if area not in {"raw", "processed"}:
        raise ValueError(f"Expected a raw or upstream processed input: {relative_path}")
    input_paths[key] = project_file(relative_path, area)
output_paths = {key: project_file(path, "processed", must_exist=False) for key, path in OUTPUTS.items()}
if len(set(output_paths.values())) != len(output_paths):
    raise ValueError("Each output needs a distinct file path.")
if set(input_paths.values()) & set(output_paths.values()):
    raise ValueError("An output must not overwrite one of its inputs.")
existing = [str(path) for path in output_paths.values() if path.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")

if set(EXPECTED_INPUT_SHA256) != set(INPUTS):
    raise ValueError("Every input must have its own expected SHA-256.")
input_fingerprints = {key: verify_input_sha256(path, EXPECTED_INPUT_SHA256[key])
                      for key, path in input_paths.items()}
inputs = {key: INPUT_READERS[key](path) for key, path in input_paths.items()}
for key, frame in inputs.items():
    print({"input": key, "data_path": INPUTS[key], "rows": len(frame),
           "schema": {column: str(dtype) for column, dtype in frame.dtypes.items()},
           "sha256": input_fingerprints[key]})

{'input': 'traffic_2024', 'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet', 'rows': 7776981, 'schema': {'rent_time': 'object', 'rent_station': 'object', 'rent_station_id': 'object', 'return_station': 'object', 'return_station_id': 'object', 'rent': 'object', 'infodate': 'object'}, 'sha256': '5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119'}
{'input': 'traffic_2025', 'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet', 'rows': 10253912, 'schema': {'rent_time': 'object', 'rent_station': 'object', 'rent_station_id': 'object', 'return_station': 'object', 'return_station_id': 'object', 'rent': 'object', 'infodate': 'object'}, 'sha256': '9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9'}
{'input': 'calendar_2024', 'data_path': 'data/processed/calendar_2024.csv', 'rows': 366, 'schema': {'date': 'object', 'day_of_week': 'object', 'weekday': 'object', 'semester': 'object', 'day_off_reason': 'object'}, 'sha25

## 1. 合併交易資料與基本核對

確認站點 ID 都是文字、只有 9 碼 ID 與 `-1` 兩種值；`infodate` 與借車日期一致；借車時間都是整點。

In [3]:
OUTSIDE_ID = "區外"
THRESHOLD = 30
FAILED_RENTAL_MAX_SECONDS = 60

trips_raw = pd.concat([inputs["traffic_2024"], inputs["traffic_2025"]], ignore_index=True)
for column in TRAFFIC_COLUMNS:
    trips_raw[column] = trips_raw[column].astype("string")
print(f"合併後旅次：{len(trips_raw):,} 筆")

assert trips_raw[TRAFFIC_COLUMNS].notna().all().all(), "Unexpected missing values."
for column in ["rent_station_id", "return_station_id"]:
    assert trips_raw[column].str.fullmatch(r"\d{9}|-1").all(), f"Unexpected ID format in {column}"
assert (trips_raw["infodate"] == trips_raw["rent_time"].str.slice(0, 10)).all(), "infodate differs from rent date."
assert trips_raw["rent_time"].str.slice(14).eq("00:00").all(), "rent_time is not on the hour."
print("ID 格式、infodate = 借車日期、整點借車時間：全部通過")

合併後旅次：18,030,893 筆


ID 格式、infodate = 借車日期、整點借車時間：全部通過


## 2. 站點 ID 與站名

範圍內站點以 `station_id` 辨識。下面列出同一個 ID 對應到多個站名的情形（若有，需要組員確認）。

In [4]:
station_names = pd.concat([
    trips_raw[["rent_station_id", "rent_station"]].set_axis(["station_id", "station_name"], axis=1),
    trips_raw[["return_station_id", "return_station"]].set_axis(["station_id", "station_name"], axis=1),
])
station_names = station_names[station_names["station_id"] != "-1"]
name_counts = (station_names.value_counts().rename("n_records").reset_index())
multi_name = name_counts[name_counts.duplicated("station_id", keep=False)]
print(f"範圍內站點 ID 數：{name_counts['station_id'].nunique()}")
print(f"對應多個站名的 ID 數：{multi_name['station_id'].nunique()}")
display(multi_name)
assert multi_name.empty, "Some IDs map to multiple names; confirm with the team before continuing."

STATION_NAME = dict(zip(name_counts["station_id"], name_counts["station_name"]))
STATIONS = sorted(STATION_NAME)
assert len(STATIONS) == 109

範圍內站點 ID 數：109
對應多個站名的 ID 數：0


,station_id,station_name,n_records


## 3. 排除借車失敗（同站借還且騎乘時間 ≤ 1 分鐘）

`rent` 為騎乘時間 HH:MM:SS，少數超過 99 小時（例如 `160:16:27`），所以以冒號拆開換算秒數，不假設小時只有兩位數。

In [5]:
same_station = trips_raw["rent_station_id"] == trips_raw["return_station_id"]
duration_parts = trips_raw.loc[same_station, "rent"].str.split(":", expand=True).astype(int)
assert duration_parts.shape[1] == 3
same_station_seconds = duration_parts[0] * 3600 + duration_parts[1] * 60 + duration_parts[2]
failed = pd.Series(False, index=trips_raw.index)
failed.loc[same_station_seconds.index] = same_station_seconds <= FAILED_RENTAL_MAX_SECONDS

n_same = int(same_station.sum())
n_failed = int(failed.sum())
print(f"同站借還：{n_same:,} 筆，其中騎乘 ≤ 1 分鐘（借車失敗）：{n_failed:,} 筆，已排除")
print("借車失敗的月份分佈：")
display(trips_raw.loc[failed, "infodate"].str.slice(0, 7).value_counts().sort_index().rename("n_trips").to_frame().T)
trips = trips_raw.loc[~failed].copy()

同站借還：473,375 筆，其中騎乘 ≤ 1 分鐘（借車失敗）：2,165 筆，已排除
借車失敗的月份分佈：


infodate,2025-08,2025-09,2025-10,2025-11,2025-12
n_trips,95,547,451,591,481


## 4. 排除從範圍外借車的旅次

依組員決定，「區外」只作為終點。借車站為 `-1` 的旅次（外借台大還）不進入轉移機率與借車到達率的計算。

In [6]:
outside_origin = trips["rent_station_id"] == "-1"
print(f"借車站在範圍外：{int(outside_origin.sum()):,} 筆，已排除")
trips = trips.loc[~outside_origin].copy()
trips["origin_id"] = trips["rent_station_id"]
trips["dest_id"] = trips["return_station_id"].where(trips["return_station_id"] != "-1", OUTSIDE_ID)
trips["hour"] = trips["rent_time"].str.slice(11, 13).astype("int8")
print(f"剩餘：{len(trips):,} 筆（還到區外 {int((trips['dest_id'] == OUTSIDE_ID).sum()):,} 筆）")

借車站在範圍外：3,920,501 筆，已排除


剩餘：14,108,227 筆（還到區外 4,261,858 筆）


## 5. 合併日曆並排除特殊日期

特殊日期 = 星期一到五且 `weekday = 0`（國定假日、颱風假、校慶等），或星期六、日且 `weekday = 1`（補班）。

In [7]:
calendar = pd.concat([inputs["calendar_2024"], inputs["calendar_2025"]], ignore_index=True)
calendar["day_of_week"] = calendar["day_of_week"].astype("int8")
calendar["weekday"] = calendar["weekday"].astype("int8")
calendar["semester"] = calendar["semester"].astype("int8")
assert not calendar["date"].duplicated().any()
assert calendar["date"].tolist() == pd.date_range("2024-01-01", "2025-12-31").strftime("%Y-%m-%d").tolist()
assert calendar["semester"].isin([0, 1]).all() and calendar["weekday"].isin([0, 1]).all()

calendar["period"] = np.where(calendar["semester"] == 1, "學期", "假期")
calendar["special"] = (((calendar["day_of_week"] <= 5) & (calendar["weekday"] == 0))
                       | ((calendar["day_of_week"] >= 6) & (calendar["weekday"] == 1)))

before = len(trips)
trips = trips.merge(calendar[["date", "period", "day_of_week", "special"]],
                    left_on="infodate", right_on="date", how="left", validate="many_to_one")
assert len(trips) == before and trips["period"].notna().all(), "Every trip date must match the calendar once."

trips_per_date = trips.groupby("infodate").size()
raw_trips_per_date = trips_raw.groupby("infodate").size()
excluded_dates = calendar.loc[calendar["special"], ["date", "day_of_week", "weekday", "period", "day_off_reason"]].copy()
excluded_dates["n_trips_used_otherwise"] = excluded_dates["date"].map(trips_per_date).fillna(0).astype(int)
excluded_dates["n_trips_raw"] = excluded_dates["date"].map(raw_trips_per_date).fillna(0).astype(int)
print(f"特殊日期：{len(excluded_dates)} 天；排除旅次 {excluded_dates['n_trips_used_otherwise'].sum():,} 筆"
      "（經第 3、4 步篩選後、原本會進入計算的旅次；n_trips_raw 為該日全部原始旅次）")
with pd.option_context("display.max_rows", 100):
    display(excluded_dates.reset_index(drop=True))

trips = trips.loc[~trips["special"]].drop(columns=["date", "special"])
trips["day_type"] = np.where(trips["day_of_week"] <= 5, "平日", "週末")
print(f"進入機率計算的旅次：{len(trips):,} 筆")

特殊日期：36 天；排除旅次 315,126 筆（經第 3、4 步篩選後、原本會進入計算的旅次；n_trips_raw 為該日全部原始旅次）


,date,day_of_week,weekday,period,day_off_reason,n_trips_used_otherwise,n_trips_raw
0,2024-01-01,1,0,假期,開國紀念日,6288,8863
1,2024-02-08,4,0,假期,春節,2023,2893
2,2024-02-09,5,0,假期,春節,1304,1885
3,2024-02-12,1,0,假期,春節,3587,5128
4,2024-02-13,2,0,假期,春節,4304,6197
5,2024-02-14,3,0,假期,春節,5438,7692
6,2024-02-17,6,1,學期,補班,8844,12071
7,2024-02-28,3,0,學期,和平紀念日,8800,12119
8,2024-04-04,4,0,學期,兒童節及清明節,7006,9557
9,2024-04-05,5,0,學期,兒童節及清明節,7805,10707


進入機率計算的旅次：13,793,101 筆


## 6. 條件轉移機率與備援機率

- `n_trips`：(p, w, h, i) 中還到 j 的筆數；`n_origin_total`：(p, w, h, i) 從 i 借出的總筆數。
- `prob = n_trips / n_origin_total`；`n_origin_total < 30` 時為 NA，`below_threshold = True`。
- `prob_fallback`：以 (p, 平日／週末, h, i) 計算的同一機率；該組借出總筆數 < 30 時也為 NA。
- 長表只存「在備援格（p, 平日／週末, h, i）中至少出現一次」的 (i, j)。每個 (p, w, h, i, j) 的旅次都屬於它的備援格，所以不會漏掉任何有旅次的組合，同時讓備援機率能在原本 0 筆的格子使用。

In [8]:
spec_keys = ["period", "day_of_week", "hour", "origin_id"]
fb_keys = ["period", "day_type", "hour", "origin_id"]

spec_pairs = trips.groupby(spec_keys + ["dest_id"], observed=True).size().rename("n_trips").reset_index()
spec_totals = trips.groupby(spec_keys, observed=True).size().rename("n_origin_total").reset_index()
fb_pairs = trips.groupby(fb_keys + ["dest_id"], observed=True).size().rename("n_fb").reset_index()
fb_totals = trips.groupby(fb_keys, observed=True).size().rename("n_fb_total").reset_index()

day_types = pd.DataFrame({"day_type": ["平日"] * 5 + ["週末"] * 2,
                          "day_of_week": np.arange(1, 8, dtype="int8")})
long = (fb_pairs.merge(fb_totals, on=fb_keys, validate="many_to_one")
        .merge(day_types, on="day_type", validate="many_to_many"))
long = (long.merge(spec_pairs, on=spec_keys + ["dest_id"], how="left", validate="one_to_one")
            .merge(spec_totals, on=spec_keys, how="left", validate="many_to_one"))
long[["n_trips", "n_origin_total"]] = long[["n_trips", "n_origin_total"]].fillna(0)
assert long["n_trips"].sum() == len(trips), "Every trip must appear exactly once."

long["below_threshold"] = long["n_origin_total"] < THRESHOLD
long["prob"] = np.where(long["below_threshold"], np.nan, long["n_trips"] / long["n_origin_total"])
long["prob_fallback"] = np.where(long["n_fb_total"] < THRESHOLD, np.nan, long["n_fb"] / long["n_fb_total"])

period_type = pd.CategoricalDtype(["學期", "假期"])
transition_probs = pd.DataFrame({
    "period": long["period"].astype(period_type),
    "day_of_week": long["day_of_week"].astype("int8"),
    "hour": long["hour"].astype("int8"),
    "origin_id": long["origin_id"].astype("string"),
    "origin_name": long["origin_id"].map(STATION_NAME).astype("string"),
    "dest_id": long["dest_id"].astype("string"),
    "dest_name": long["dest_id"].map(STATION_NAME | {OUTSIDE_ID: OUTSIDE_ID}).astype("string"),
    "n_trips": long["n_trips"].astype("int32"),
    "n_origin_total": long["n_origin_total"].astype("int32"),
    "prob": long["prob"].astype("float64"),
    "below_threshold": long["below_threshold"].astype("bool"),
    "prob_fallback": long["prob_fallback"].astype("float64"),
}).sort_values(["period", "day_of_week", "hour", "origin_id", "dest_id"], ignore_index=True)

assert not transition_probs.duplicated(["period", "day_of_week", "hour", "origin_id", "dest_id"]).any()
assert transition_probs[["origin_name", "dest_name"]].notna().all().all()
assert set(transition_probs["origin_id"]) <= set(STATIONS)
assert set(transition_probs["dest_id"]) <= set(STATIONS) | {OUTSIDE_ID}
print(f"transition_probs：{len(transition_probs):,} 列")
display(transition_probs.head(10))

transition_probs：2,308,664 列


,period,day_of_week,hour,origin_id,origin_name,dest_id,dest_name,n_trips,n_origin_total,prob,below_threshold,prob_fallback
0,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101001,YouBike2.0_捷運科技大樓站,1,399,0.002506,False,0.002535
1,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101002,YouBike2.0_復興南路二段273號前,3,399,0.007519,False,0.005070
2,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101003,YouBike2.0_國北教大實小東側門,3,399,0.007519,False,0.005070
3,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101004,YouBike2.0_和平公園東側,7,399,0.017544,False,0.013519
4,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101005,YouBike2.0_辛亥復興路口西北側,16,399,0.040100,False,0.027461
5,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101006,YouBike2.0_復興南路二段280號前,2,399,0.005013,False,0.002112
6,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101007,YouBike2.0_復興南路二段340巷口,6,399,0.015038,False,0.015209
7,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101008,YouBike2.0_新生南路三段52號前,0,399,0.000000,False,0.002535
8,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101009,YouBike2.0_新生南路三段66號前,1,399,0.002506,False,0.001690
9,學期,1,0,500101001,YouBike2.0_捷運科技大樓站,500101010,YouBike2.0_新生南路三段82號前,0,399,0.000000,False,0.000422


## 7. 樣本數門檻統計

所有可能的 (p, w, h, i) 組合共 2 × 7 × 24 × 109 = 36,624 個，包含完全沒有借車的組合（借出 0 筆也算低於門檻）。

In [9]:
all_cells = pd.MultiIndex.from_product(
    [["學期", "假期"], range(1, 8), range(24), STATIONS], names=spec_keys).to_frame(index=False)
cell_totals = all_cells.merge(spec_totals.astype({"day_of_week": "int64", "hour": "int64"}),
                              on=spec_keys, how="left").fillna({"n_origin_total": 0})
cell_totals["below"] = cell_totals["n_origin_total"] < THRESHOLD
cell_totals["day_type"] = np.where(cell_totals["day_of_week"] <= 5, "平日", "週末")
fb_cells = (cell_totals[fb_keys].drop_duplicates()
            .merge(fb_totals.astype({"hour": "int64"}), on=fb_keys, how="left").fillna({"n_fb_total": 0}))
fb_cells["fb_below"] = fb_cells["n_fb_total"] < THRESHOLD
cell_totals = cell_totals.merge(fb_cells[fb_keys + ["fb_below"]], on=fb_keys, validate="many_to_one")

n_cells = len(cell_totals)
n_below = int(cell_totals["below"].sum())
n_zero = int((cell_totals["n_origin_total"] == 0).sum())
n_both = int((cell_totals["below"] & cell_totals["fb_below"]).sum())
print(f"(p, w, h, i) 共 {n_cells:,} 個；低於門檻 {n_below:,} 個（{n_below / n_cells:.1%}），其中借出 0 筆 {n_zero:,} 個")
print(f"備援格 (p, 平日／週末, h, i) 共 {len(fb_cells):,} 個；低於門檻 {int(fb_cells['fb_below'].sum()):,} 個")
print(f"原本低於門檻、備援也低於門檻（模擬時仍為 NA）：{n_both:,} 個（{n_both / n_cells:.1%}）")

by_hour = (cell_totals.groupby("hour")
           .agg(below=("below", "sum"), both_na=("fb_below", lambda s: int((s & cell_totals.loc[s.index, "below"]).sum())))
           .assign(share_below=lambda d: d["below"] / (n_cells / 24)))
print("\n各小時低於門檻的組合數（每小時共 1,526 個）：")
display(by_hour.T.round(2))

by_period = cell_totals.groupby(["period"])["below"].mean().round(3)
print("各時期低於門檻比例：", by_period.to_dict())

by_station = (cell_totals.groupby("origin_id")
              .agg(below=("below", "sum"), both_na=("fb_below", lambda s: int((s & cell_totals.loc[s.index, "below"]).sum())))
              .assign(origin_name=lambda d: d.index.map(STATION_NAME),
                      share_below=lambda d: d["below"] / 336)
              .sort_values("below", ascending=False))
print("\n低於門檻最多的 15 站（每站共 336 個組合）：")
display(by_station.head(15).round(2))
print(f"每個組合都達門檻（0 個低於門檻）的站：{int((by_station['below'] == 0).sum())} 站")

(p, w, h, i) 共 36,624 個；低於門檻 8,126 個（22.2%），其中借出 0 筆 292 個
備援格 (p, 平日／週末, h, i) 共 10,464 個；低於門檻 1,046 個
原本低於門檻、備援也低於門檻（模擬時仍為 NA）：3,112 個（8.5%）

各小時低於門檻的組合數（每小時共 1,526 個）：


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
below,523.00,960.00,1240.00,1398.00,1420.00,1055.00,543.00,216.00,71.00,35.00,...,23.00,18.00,13.00,8.00,12.00,37.00,55.00,60.00,123.00,248.00
both_na,71.00,242.00,434.00,753.00,797.00,454.00,133.00,60.00,21.00,15.00,...,9.00,7.00,7.00,7.00,9.00,7.00,15.00,11.00,9.00,17.00
share_below,0.34,0.63,0.81,0.92,0.93,0.69,0.36,0.14,0.05,0.02,...,0.02,0.01,0.01,0.01,0.01,0.02,0.04,0.04,0.08,0.16


各時期低於門檻比例： {'假期': 0.279, '學期': 0.164}

低於門檻最多的 15 站（每站共 336 個組合）：


,below,both_na,origin_name,share_below
origin_id,,,,
500101265,227,200,YouBike2.0_羅斯福路三段283巷,0.68
500119008,189,95,YouBike2.0_臺大檔案展示館,0.56
500119080,160,90,YouBike2.0_臺大小小福西南側,0.48
500119077,156,94,YouBike2.0_臺大博雅館西側,0.46
500119082,148,63,YouBike2.0_臺大四號館東北側,0.44
500119078,134,56,YouBike2.0_臺大森林館北側,0.40
500119093,132,62,YouBike2.0_臺大禮賢樓西側,0.39
500119069,131,46,YouBike2.0_臺大思亮館東南側,0.39
500119063,129,75,YouBike2.0_臺大第二行政大樓南側,0.38


每個組合都達門檻（0 個低於門檻）的站：0 站


## 8. 每站每小時平均借出量（借車到達率）

營運起日 = 每小時流量檔中 `in_service` 第一次為 1 的日期，與該站第一筆交易（借或還、清理前）日期取較晚者。`n_days` = 該 (時期, 星期) 扣除特殊日期後，營運起日當天或之後的天數。

In [10]:
flow = inputs["hourly_flow"]
flow_service = flow.loc[flow["in_service"] == 1].assign(date=lambda d: d["datetime"].str.slice(0, 10))
in_service_start = flow_service.groupby("station_id")["date"].min()
# in_service never returns to 0 after a station starts (checked in its documentation); confirm here.
service_hours = flow.assign(date=flow["datetime"].str.slice(0, 10)).merge(
    in_service_start.rename("start"), left_on="station_id", right_index=True)
assert (service_hours.loc[service_hours["date"] >= service_hours["start"], "in_service"] == 1).all()

first_trip = pd.concat([
    trips_raw[["rent_station_id", "infodate"]].set_axis(["station_id", "date"], axis=1),
    trips_raw[["return_station_id", "infodate"]].set_axis(["station_id", "date"], axis=1),
]).query("station_id != '-1'").groupby("station_id")["date"].min()

service = pd.DataFrame({"in_service_start": in_service_start, "first_trip": first_trip})
assert set(service.index) == set(STATIONS) and service.notna().all().all()
service["service_start"] = service.max(axis=1)
late = service[service["service_start"] > "2024-01-01"].assign(station_name=lambda d: d.index.map(STATION_NAME))
print("營運起日晚於 2024-01-01 的站：")
display(late)

valid_days = calendar.loc[~calendar["special"], ["date", "period", "day_of_week"]]
n_days = (valid_days.merge(service["service_start"].rename_axis("origin_id").reset_index(), how="cross")
          .query("date >= service_start")
          .groupby(["period", "day_of_week", "origin_id"]).size().rename("n_days").reset_index())

rent_counts = trips.groupby(spec_keys, observed=True).size().rename("n_trips_total").reset_index()
rent_rates = (all_cells
              .merge(rent_counts.astype({"day_of_week": "int64", "hour": "int64"}), on=spec_keys, how="left")
              .merge(n_days.astype({"day_of_week": "int64"}), on=["period", "day_of_week", "origin_id"], how="left"))
rent_rates[["n_trips_total", "n_days"]] = rent_rates[["n_trips_total", "n_days"]].fillna(0)
rent_rates["mean_rentals_per_day"] = np.where(rent_rates["n_days"] > 0,
                                              rent_rates["n_trips_total"] / rent_rates["n_days"], np.nan)
rent_rates = pd.DataFrame({
    "period": rent_rates["period"].astype(period_type),
    "day_of_week": rent_rates["day_of_week"].astype("int8"),
    "hour": rent_rates["hour"].astype("int8"),
    "origin_id": rent_rates["origin_id"].astype("string"),
    "n_trips_total": rent_rates["n_trips_total"].astype("int32"),
    "n_days": rent_rates["n_days"].astype("int32"),
    "mean_rentals_per_day": rent_rates["mean_rentals_per_day"].astype("float64"),
}).sort_values(spec_keys, ignore_index=True)

assert len(rent_rates) == 2 * 7 * 24 * 109
assert not rent_rates.duplicated(spec_keys).any()
assert rent_rates["n_trips_total"].sum() == len(trips)
# No trips may come from a station on days before its service start.
assert (rent_rates.loc[rent_rates["n_days"] == 0, "n_trips_total"] == 0).all()
print(f"rent_rates：{len(rent_rates):,} 列；n_days = 0 的列：{int((rent_rates['n_days'] == 0).sum())}")
display(rent_rates.head(10))

營運起日晚於 2024-01-01 的站：


,in_service_start,first_trip,service_start,station_name
station_id,,,,
500101234,2024-03-19,2024-03-19,2024-03-19,YouBike2.0_和平東路二段90巷
500101240,2024-09-20,2024-09-20,2024-09-20,YouBike2.0_和平東路二段76巷
500101250,2024-12-13,2024-12-13,2024-12-13,YouBike2.0_國立臺北教育大學創意館
500101265,2025-10-10,2025-10-10,2025-10-10,YouBike2.0_羅斯福路三段283巷
500106135,2024-10-15,2024-10-15,2024-10-15,YouBike2.0_羅斯福路四段108巷口
500119059,2024-01-02,2024-06-12,2024-06-12,YouBike2.0_臺大共同教室北側
500119092,2024-08-13,2024-08-13,2024-08-13,YouBike2.0_臺大管理學院一館
500119093,2024-08-16,2024-08-16,2024-08-16,YouBike2.0_臺大禮賢樓西側
500119094,2024-08-13,2024-08-13,2024-08-13,YouBike2.0_臺大大一女餐廳廣場


rent_rates：36,624 列；n_days = 0 的列：48


,period,day_of_week,hour,origin_id,n_trips_total,n_days,mean_rentals_per_day
0,學期,1,0,500101001,399,65,6.138462
1,學期,1,0,500101002,130,65,2.000000
2,學期,1,0,500101003,18,65,0.276923
3,學期,1,0,500101004,53,65,0.815385
4,學期,1,0,500101005,68,65,1.046154
5,學期,1,0,500101006,80,65,1.230769
6,學期,1,0,500101007,57,65,0.876923
7,學期,1,0,500101008,45,65,0.692308
8,學期,1,0,500101009,46,65,0.707692
9,學期,1,0,500101010,138,65,2.123077


## 9. 檢查

1. 每個未標 NA 的 (p, w, h, i)：機率加總 = 1（誤差 < 1e-9）；備援機率同樣檢查。
2. 14 個 (時期, 星期) 組合的天數 + 特殊日期數 = 資料涵蓋的總天數。
3. 隨機抽 3 個 (p, w, h, i)，直接從原始交易表重新篩選、手算機率，和輸出比對。

In [11]:
# Check 1: probabilities sum to 1
valid = transition_probs.loc[~transition_probs["below_threshold"]]
sums = valid.groupby(spec_keys, observed=True)["prob"].sum()
max_err = float((sums - 1).abs().max())
print(f"檢查 1：{len(sums):,} 個未標 NA 的 (p, w, h, i)，機率加總與 1 的最大誤差 = {max_err:.2e}")
assert max_err < 1e-9
fb_sums = (transition_probs.dropna(subset=["prob_fallback"])
           .assign(day_type=lambda d: np.where(d["day_of_week"] <= 5, "平日", "週末"))
           .groupby(["period", "day_of_week", "hour", "origin_id"], observed=True)["prob_fallback"].sum())
fb_err = float((fb_sums - 1).abs().max())
print(f"         備援機率：{len(fb_sums):,} 個 (p, w, h, i)，最大誤差 = {fb_err:.2e}")
assert fb_err < 1e-9
assert (transition_probs.loc[transition_probs["below_threshold"], "prob"].isna().all()
        and transition_probs.loc[~transition_probs["below_threshold"], "prob"].notna().all())

檢查 1：28,498 個未標 NA 的 (p, w, h, i)，機率加總與 1 的最大誤差 = 1.11e-16


         備援機率：33,512 個 (p, w, h, i)，最大誤差 = 1.11e-16


In [12]:
# Check 2: day counts
calendar_days = (calendar.loc[~calendar["special"]].groupby(["period", "day_of_week"]).size()
                 .rename("n_days").reset_index())
n_special = int(calendar["special"].sum())
n_total_days = len(calendar)
first_date, last_date = trips_raw["infodate"].min(), trips_raw["infodate"].max()
span_days = (pd.Timestamp(last_date) - pd.Timestamp(first_date)).days + 1
print(f"檢查 2：14 個 (時期, 星期) 的天數：")
display(calendar_days.pivot(index="period", columns="day_of_week", values="n_days"))
print(f"14 組天數合計 {calendar_days['n_days'].sum()} + 特殊日期 {n_special} = {calendar_days['n_days'].sum() + n_special}；"
      f"資料涵蓋 {first_date}～{last_date} 共 {span_days} 天（其中沒有旅次的日期："
      f"{sorted(set(calendar['date']) - set(trips_raw['infodate']))}）")
assert len(calendar_days) == 14
assert calendar_days["n_days"].sum() + n_special == span_days == n_total_days
# For stations operating since 2024-01-01, n_days equals the calendar day count.
full_stations = service.index[service["service_start"] == "2024-01-01"]
check_days = (rent_rates[rent_rates["origin_id"].isin(full_stations)]
              .drop_duplicates(["period", "day_of_week", "origin_id"])
              .astype({"period": str, "day_of_week": "int64"})
              .merge(calendar_days, on=["period", "day_of_week"], suffixes=("", "_calendar")))
assert (check_days["n_days"] == check_days["n_days_calendar"]).all()
print(f"自 2024-01-01 起營運的 {len(full_stations)} 站，n_days 都等於日曆天數")

檢查 2：14 個 (時期, 星期) 的天數：


day_of_week,1,2,3,4,5,6,7
period,,,,,,,
假期,34,35,33,33,34,34,35
學期,65,67,66,62,60,68,69


14 組天數合計 695 + 特殊日期 36 = 731；資料涵蓋 2024-01-01～2025-12-31 共 731 天（其中沒有旅次的日期：['2024-07-25', '2024-10-03']）
自 2024-01-01 起營運的 97 站，n_days 都等於日曆天數


In [13]:
# Check 3: recompute 3 random (p, w, h, i) directly from the raw trip table
rng = np.random.default_rng(5166)
valid_cells = valid[spec_keys].drop_duplicates().reset_index(drop=True)
sample_cells = valid_cells.iloc[rng.choice(len(valid_cells), size=3, replace=False)]

cal_lookup = calendar.set_index("date")
for _, cell in sample_cells.iterrows():
    p, w, h, i = cell["period"], int(cell["day_of_week"]), int(cell["hour"]), cell["origin_id"]
    dates = cal_lookup.index[(cal_lookup["period"] == p) & (cal_lookup["day_of_week"] == w) & ~cal_lookup["special"]]
    raw = trips_raw[(trips_raw["rent_station_id"] == i)
                    & trips_raw["infodate"].isin(dates)
                    & (trips_raw["rent_time"].str.slice(11, 13) == f"{h:02d}")]
    secs = raw["rent"].str.split(":", expand=True).astype(int)
    secs = secs[0] * 3600 + secs[1] * 60 + secs[2]
    raw = raw[~((raw["return_station_id"] == i) & (secs <= 60))]
    manual = (raw["return_station_id"].replace("-1", OUTSIDE_ID).value_counts() / len(raw))
    out = valid[(valid["period"] == p) & (valid["day_of_week"] == w) & (valid["hour"] == h)
                & (valid["origin_id"] == i)].set_index("dest_id")["prob"]
    out = out[out > 0]
    diff = manual.reindex(out.index.union(manual.index), fill_value=0) - out.reindex(out.index.union(manual.index), fill_value=0)
    print(f"檢查 3：({p}, 星期{w}, {h} 時, {STATION_NAME[i]})：手算 {len(raw)} 筆、輸出 n_origin_total = "
          f"{int(valid.loc[(valid['period'] == p) & (valid['day_of_week'] == w) & (valid['hour'] == h) & (valid['origin_id'] == i), 'n_origin_total'].iloc[0])}；"
          f"終點數 {len(manual)} / {len(out)}；最大機率差 {float(diff.abs().max()):.2e}")
    assert float(diff.abs().max()) < 1e-12

檢查 3：(假期, 星期1, 21 時, YouBike2.0_臺大萬才館前)：手算 71 筆、輸出 n_origin_total = 71；終點數 19 / 19；最大機率差 0.00e+00


檢查 3：(假期, 星期7, 12 時, YouBike2.0_臺大新生教室南側)：手算 148 筆、輸出 n_origin_total = 148；終點數 48 / 48；最大機率差 0.00e+00


檢查 3：(學期, 星期4, 2 時, YouBike2.0_臺大男七舍前)：手算 36 筆、輸出 n_origin_total = 36；終點數 14 / 14；最大機率差 0.00e+00


## 10. 存檔（Parquet，pyarrow）並讀回核對

In [14]:
outputs = {"transition_probs": transition_probs, "rent_rates": rent_rates}
assert set(outputs) == set(OUTPUTS), "Produced datasets must match the declared output paths."

verification = {}
for key, frame in outputs.items():
    path = output_paths[key]
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        raise FileExistsError(path)  # never overwrite silently during a rerun
    frame.to_parquet(path, engine="pyarrow", index=False)
    reread = pd.read_parquet(path, engine="pyarrow")
    assert list(reread.columns) == list(frame.columns), f"Schema changed while saving {key}."
    assert len(reread) == len(frame), f"Row count changed while saving {key}."
    dtype_before = {c: str(t) for c, t in frame.dtypes.items()}
    dtype_after = {c: str(t) for c, t in reread.dtypes.items()}
    assert dtype_before == dtype_after, f"Dtypes changed while saving {key}: {dtype_before} vs {dtype_after}"
    assert list(reread["period"].cat.categories) == ["學期", "假期"]
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), reread)
    verification[key] = {"data_path": OUTPUTS[key], "rows": len(reread),
                         "schema": dtype_after, "sha256": sha256_file(path),
                         "size_MB": round(path.stat().st_size / 1e6, 2),
                         "status": "read-back checks passed (rows, column order, dtypes, values)"}
    print(verification[key])
assert all(sha256_file(path) == input_fingerprints[key] for key, path in input_paths.items()), "An input changed."

{'data_path': 'data/processed/youbike_transition/transition_probs.parquet', 'rows': 2308664, 'schema': {'period': 'category', 'day_of_week': 'int8', 'hour': 'int8', 'origin_id': 'string', 'origin_name': 'string', 'dest_id': 'string', 'dest_name': 'string', 'n_trips': 'int32', 'n_origin_total': 'int32', 'prob': 'float64', 'below_threshold': 'bool', 'prob_fallback': 'float64'}, 'sha256': '13e3f831cd6f5cb5324ce1a7a8aad3d1500899db33d82321ff9457139b48affc', 'size_MB': 14.4, 'status': 'read-back checks passed (rows, column order, dtypes, values)'}
{'data_path': 'data/processed/youbike_transition/rent_rates.parquet', 'rows': 36624, 'schema': {'period': 'category', 'day_of_week': 'int8', 'hour': 'int8', 'origin_id': 'string', 'n_trips_total': 'int32', 'n_days': 'int32', 'mean_rentals_per_day': 'float64'}, 'sha256': 'cb428e819d279553dd3ba681f1ba1e36635ed48be9833547c42bfe86e377e41d', 'size_MB': 0.19, 'status': 'read-back checks passed (rows, column order, dtypes, values)'}


## 11. `get_matrix()`：讀取某時段的轉移矩陣

函式定義在 [transition_matrix.py](transition_matrix.py)（模擬程式可直接 `from transition_matrix import get_matrix`），這裡匯入並測試：列 = 109 個借車站、欄 = 109 站 + 區外；達門檻的列加總為 1。

In [15]:
sys.path.insert(0, str(PROJECT_ROOT / "data" / "code"))
from transition_matrix import get_matrix

for args in [("學期", 1, 8), ("假期", 7, 3)]:
    for fb in (False, True):
        m = get_matrix(*args, fallback=fb)
        row_sums = m.sum(axis=1, min_count=1)
        complete = row_sums.notna()
        assert m.shape == (109, 110)
        assert (m.loc[complete].notna().all(axis=1)).all() and ((row_sums[complete] - 1).abs() < 1e-9).all()
        print(f"get_matrix{args}, fallback={fb}: shape {m.shape}，可用的列 {int(complete.sum())} / 109")

get_matrix('學期', 1, 8), fallback=False: shape (109, 110)，可用的列 109 / 109
get_matrix('學期', 1, 8), fallback=True: shape (109, 110)，可用的列 109 / 109
get_matrix('假期', 7, 3), fallback=False: shape (109, 110)，可用的列 5 / 109


get_matrix('假期', 7, 3), fallback=True: shape (109, 110)，可用的列 13 / 109


## 12. 流動模式摘要

以學期中星期一 8 時與 17 時為例，看捷運公館站與捷運科技大樓站借出的車主要還到哪裡，以及範圍內各站借出後還到區外的比例。

In [16]:
def top_destinations(period, dow, hour, origin_keyword, k=5):
    m = get_matrix(period, dow, hour, fallback=False)
    rows = [s for s in m.index if origin_keyword in STATION_NAME[s]]
    out = []
    for s in rows:
        n = int(transition_probs.query("period == @period and day_of_week == @dow and hour == @hour and origin_id == @s")["n_origin_total"].max())
        top = m.loc[s].dropna().sort_values(ascending=False).head(k)
        out.append(pd.DataFrame({"origin": STATION_NAME[s].removeprefix("YouBike2.0_"), "n_origin_total": n,
                                 "dest": [STATION_NAME.get(d, d).removeprefix("YouBike2.0_") for d in top.index],
                                 "prob": top.values.round(3)}))
    return pd.concat(out, ignore_index=True)

print("學期、星期一 8 時：")
display(top_destinations("學期", 1, 8, "捷運公館站"))
display(top_destinations("學期", 1, 8, "捷運科技大樓站"))
print("學期、星期一 17 時：")
display(top_destinations("學期", 1, 17, "捷運公館站"))

campus = [s for s in STATIONS if "臺大" in STATION_NAME[s]]
for hour in (8, 17):
    m = get_matrix("學期", 1, hour, fallback=False)
    to_mrt = m.loc[m.index.isin(campus), [s for s in m.columns if "捷運" in STATION_NAME.get(s, "")]].sum(axis=1)
    print(f"學期、星期一 {hour} 時：校內（站名含「臺大」）{int(to_mrt.notna().sum())} 個可用站，還到捷運站的機率中位數 {to_mrt.median():.3f}；"
          f"全部可用站還到區外的機率中位數 {m[OUTSIDE_ID].median():.3f}")

學期、星期一 8 時：


,origin,n_origin_total,dest,prob
0,捷運公館站(2號出口),3989,區外,0.297
1,捷運公館站(2號出口),3989,臺大男一舍前,0.056
2,捷運公館站(2號出口),3989,臺大立體機車停車場,0.054
3,捷運公館站(2號出口),3989,臺大男七舍前,0.050
4,捷運公館站(2號出口),3989,臺大社科院西側,0.043
5,捷運公館站(3號出口),1208,區外,0.099
6,捷運公館站(3號出口),1208,臺大樂學館東側,0.070
7,捷運公館站(3號出口),1208,臺大第一活動中心西南側,0.065
8,捷運公館站(3號出口),1208,臺大小福樓東側,0.061
9,捷運公館站(3號出口),1208,臺大女九舍西南側,0.060


,origin,n_origin_total,dest,prob
0,捷運科技大樓站,6115,區外,0.433
1,捷運科技大樓站,6115,辛亥復興路口西北側,0.075
2,捷運科技大樓站,6115,敦親公園,0.042
3,捷運科技大樓站,6115,臺大女九舍西南側,0.025
4,捷運科技大樓站,6115,臺大小福樓東側,0.024
5,捷運科技大樓站(台北教育大學),2058,區外,0.486
6,捷運科技大樓站(台北教育大學),2058,臺大女九舍西南側,0.027
7,捷運科技大樓站(台北教育大學),2058,臺大管理學院二館北側,0.027
8,捷運科技大樓站(台北教育大學),2058,臺大水源修齊會館,0.026
9,捷運科技大樓站(台北教育大學),2058,臺大社會系館南側,0.022


學期、星期一 17 時：


,origin,n_origin_total,dest,prob
0,捷運公館站(2號出口),6748,區外,0.441
1,捷運公館站(2號出口),6748,臺大第一活動中心西南側,0.041
2,捷運公館站(2號出口),6748,臺大男一舍前,0.032
3,捷運公館站(2號出口),6748,臺大小福樓東側,0.022
4,捷運公館站(2號出口),6748,臺大女九舍西南側,0.019
5,捷運公館站(3號出口),4772,區外,0.244
6,捷運公館站(3號出口),4772,臺大第一活動中心西南側,0.085
7,捷運公館站(3號出口),4772,臺大樂學館東側,0.044
8,捷運公館站(3號出口),4772,臺大小福樓東側,0.043
9,捷運公館站(3號出口),4772,臺大女九舍西南側,0.033


學期、星期一 8 時：校內（站名含「臺大」）72 個可用站，還到捷運站的機率中位數 0.043；全部可用站還到區外的機率中位數 0.183
學期、星期一 17 時：校內（站名含「臺大」）72 個可用站，還到捷運站的機率中位數 0.212；全部可用站還到區外的機率中位數 0.273


## 結論與限制

**執行狀態**：2026-11-10 以全新 Python 3 kernel（nbclient）由上而下執行完畢，所有 assertion 通過；以下數字都取自上方輸出。

**樣本**：18,030,893 筆旅次 → 排除借車失敗 2,165 筆（全部在 2025-08 以後，因為資料來源在那之前沒有收錄短時間同站借還）→ 排除從範圍外借車 3,920,501 筆 → 排除 36 個特殊日期的 315,126 筆 → **13,793,101 筆**進入計算。

**輸出**：
- `transition_probs.parquet`：2,308,664 列，SHA-256 `13e3f831cd6f5cb5324ce1a7a8aad3d1500899db33d82321ff9457139b48affc`。
- `rent_rates.parquet`：36,624 列，SHA-256 `cb428e819d279553dd3ba681f1ba1e36635ed48be9833547c42bfe86e377e41d`。
- 兩個檔案讀回後的列數、欄位順序、型態與數值都和存檔前相同。

**檢查**：
1. 28,498 個未標 NA 的 (p, w, h, i)，機率加總與 1 的最大誤差為 1.1e-16；備援機率同樣通過。
2. 14 個 (時期, 星期) 的天數合計 695，加上 36 個特殊日期等於 731 天（2024-01-01～2025-12-31）。
3. 隨機抽 3 個 (p, w, h, i) 從原始表手算，借出筆數與每個終點的機率完全相同。

**樣本數門檻**：
- 36,624 個 (p, w, h, i) 中有 8,126 個（22.2%）低於 30 筆，其中 292 個完全沒有借車。
- 低於門檻的組合集中在深夜：02–04 時有 81%–93% 的組合低於門檻，08–18 時只有 1%–5%。假期（27.9%）比學期（16.4%）多。
- 依站點看，集中在使用量少的校內站：羅斯福路三段283巷（2025-10 才啟用，68%）、臺大檔案展示館（56%）、臺大小小福西南側、博雅館、四號館等。
- 改用備援格後，仍有 3,112 個（8.5%）是 NA，絕大多數在 01–05 時。
- `rent_rates` 中有 48 列 `n_days = 0`（`mean_rentals_per_day` 為 NA）。這是羅斯福路三段283巷在 2025-10-10 啟用後，沒有任何「假期的星期四、五」的天數。

**流動模式**：
- 學期中星期一 8 時，從捷運公館站借出的車主要騎進校內：2 號出口以臺大男一舍、立體機車停車場、男七舍、社科院為主，3 號出口以樂學館、第一活動中心、小福樓、女九舍為主；但每一個目的地的機率都只有 4%–9%，流向很分散。
- 同一時段，公館站 1、2、4 號出口與科技大樓站借出的車有 30%–49% 騎出研究範圍。
- 17 時方向反轉：校內各站還到捷運站的機率中位數由 8 時的 4.3% 升到 21.2%；從公館站 1、4 號出口借出的車有六成以上騎出範圍。

**限制**：
- 「區外」只作為終點，所以模擬時從範圍外進入的車必須另外處理（例如另外估計還入率），本表不包含這部分。
- 營運起日只處理「晚啟用」的站，沒有處理營運期間的暫停。
- 借車失敗的篩選只在 2025-08 以後有作用（之前的資料本來就沒有這類紀錄）。
- 重跑本 notebook 前要先刪除兩個輸出檔（程式不會覆寫既有檔案）。
